# en-UG Voice Cloning — CosyVoice3

Loads the fine-tuned `en-UG` CosyVoice3 bundle from Hugging Face and runs zero-shot voice cloning:
give it a short reference clip + its transcript, and it can generate **new** speech in that voice.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or A100 if you have Colab Pro). A T4 is enough for this model (0.5B params).

## 1. Install dependencies

In [ ]:
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd CosyVoice
!pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q huggingface_hub

## 2. Authenticate with Hugging Face

The `en-UG` bundle is a private repo under the `all-lab` org. You'll need a token with read access — paste it when prompted (it won't be saved to the notebook).

In [ ]:
from getpass import getpass
import os
os.environ["HF_TOKEN"] = getpass("HF token: ")

## 3. Download the model bundle

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id="all-lab/cosyvoice3-individual-en-UG",
    token=os.environ["HF_TOKEN"],
)
print("downloaded to:", model_dir)

## 4. Load the model

In [ ]:
import sys
sys.path.insert(0, ".")
sys.path.insert(0, "third_party/Matcha-TTS")

from cosyvoice.cli.cosyvoice import CosyVoice3

model = CosyVoice3(model_dir, fp16=True)  # fp16 is a good default on T4/A100
print("sample rate:", model.sample_rate)

## 5. Get a reference clip

Upload your own short (10-15s works best) clean, single-speaker WAV clip, or run the cell below to fetch a bundled example.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick a .wav file; skip this cell to use the example below instead
prompt_wav_path = next(iter(uploaded)) if uploaded else None

In [ ]:
# Fallback example clip if you skipped the upload above (public domain speech sample)
if not prompt_wav_path:
    !wget -q -O example_ref.wav "https://www2.cs.uic.edu/~i101/SoundFiles/gettysburg10.wav"
    prompt_wav_path = "example_ref.wav"

print("using:", prompt_wav_path)

## 6. Clone the voice

**`prompt_text` must be an accurate transcript of what's actually said in the clip** — a wrong transcript degrades the clone noticeably. Type it in below.

Note the required `<|endofprompt|>` marker appended to `prompt_text` — CosyVoice3's LLM hard-requires this literal token to mark where the reference transcript ends.

In [ ]:
import torchaudio
from IPython.display import Audio, display

prompt_text = input("What is actually said in your reference clip? ") + "<|endofprompt|>"
tts_text = input("What should the cloned voice say instead? ")

results = list(model.inference_zero_shot(tts_text, prompt_text, prompt_wav_path, stream=False))
audio = results[0]["tts_speech"]
torchaudio.save("output.wav", audio, model.sample_rate)

print(f"generated {audio.shape[1] / model.sample_rate:.2f}s of audio")
display(Audio("output.wav"))

## 7. Try different voice designs

Voice identity comes entirely from the reference clip at call time, not from anything baked into the checkpoint — so a "different voice design" is just a different `prompt_wav` / `prompt_text` pair on the same loaded model. Re-run cells 5-6 with a new clip to hear another voice, or loop over several here:

In [ ]:
voices = [
    # (path_to_wav, "exact transcript of that clip"),
    # ("voiceA.wav", "..."),
    # ("voiceB.wav", "..."),
]

new_text = "Whatever sentence you want every voice to say, for a fair comparison."

for wav, text in voices:
    for i, r in enumerate(model.inference_zero_shot(new_text, text + "<|endofprompt|>", wav, stream=False)):
        out = f"out_{wav}"
        torchaudio.save(out, r["tts_speech"], model.sample_rate)
        print(wav)
        display(Audio(out))

## 8. Download your results

In [ ]:
from google.colab import files
files.download("output.wav")

---
**A few known gotchas** (all confirmed while building this pipeline):
- A harmless `no frontend is avaliable` warning may print on load — an optional text-normalization helper trying to reach a host with no internet path from some environments. Output is unaffected.
- Very short reference clips (under ~5s) have produced truncated, near-silent output in testing — stick to 10s+.
- `prompt_wav` must be a **file path**, not a pre-loaded audio tensor — CosyVoice loads and resamples it internally.